# **Autogen RoundRobin Group Agent**
- A team that runs a group chat with participants taking turns in a round-robin fashion to publish a message to all.

In [1]:
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.ollama import OllamaChatCompletionClient
import asyncio
from dotenv import load_dotenv
import os
load_dotenv()

True

## **Importing Model**

In [2]:
ollama_model_clint = OllamaChatCompletionClient(model="llama3.1")

## **Define 3 specific Agent**
- dsa solver
- code reviewer
- code editor

In [7]:
dsa_solver = AssistantAgent(
    name="Complex_DSA_solver",
    model_client=ollama_model_clint,
    description="A DSA solver",
    system_message="You give code in python to solve complex dsa problems. Give under 100 words"
)


code_reviewer = AssistantAgent(
    name="Code_Reviewer",
    model_client=ollama_model_clint,
    description="A code Reviewer",
    system_message="You review the code given by the complex_dsa_solver and make sure it is optimized. Given under 20 words. If you think that the code is fine, Please say 'TERMINATE'"
)

code_editor = AssistantAgent(
    name="Code_Editor",
    model_client=ollama_model_clint,
    description="A code Editor.",
    system_message="YOu make the code easy to understand and add comments wherever required. Give under 20 words."
)

In [10]:
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination

my_termination = TextMentionTermination(text="TERMINATE") #| MaxMessageTermination(max_messages=2)

team = RoundRobinGroupChat(
    participants=[dsa_solver, code_reviewer, code_editor],
    termination_condition=my_termination,
    max_turns= 10
)

In [11]:
from autogen_agentchat.messages import TextMessage
async def run_team():
    task = TextMessage(content="Write a python code like find a median two sorted array.", source="user")
    
    result = await team.run(task=task)
    
    for each_agent_msg in result.messages:
        print(f"{each_agent_msg.source} :-> {each_agent_msg.content}")
    
await run_team()

user :-> Write a python code like find a median two sorted array.
Complex_DSA_solver :-> Here is the Python code for finding the median of two sorted arrays using binary search:

```python
def findMedian(arr1, arr2):
    if len(arr1) > len(arr2):
        arr1, arr2 = arr2, arr1
    
    total_len = len(arr1) + len(arr2)
    half_len = total_len // 2
    
    left = 0
    right = len(arr1) - 1
    
    while True:
        i = (left + right) // 2
        j = half_len - i - 2
        
        max_left_x = float('-inf') if i < 0 else arr1[i]
        min_right_x = float('inf') if i >= len(arr1) else arr1[i+1]
        
        max_left_y = float('-inf') if j < 0 else arr2[j]
        min_right_y = float('inf') if j >= len(arr2) else arr2[j+1]
        
        if max_left_x <= min_right_y and max_left_y <= min_right_x:
            if total_len % 2 == 0:
                return (max(max_left_x, max_left_y) + min(min_right_x, min_right_y)) / 2.0
            else:
                return max(max_le

## **Built-In Termination Conditions:**

- MaxMessageTermination: Stops after a specified number of messages have been produced, including both agent and task messages.

- TextMentionTermination: Stops when specific text or string is mentioned in a message (e.g., “TERMINATE”).

- TokenUsageTermination: Stops when a certain number of prompt or completion tokens are used. This requires the agents to report token usage in their messages.

- TimeoutTermination: Stops after a specified duration in seconds.

- HandoffTermination: Stops when a handoff to a specific target is requested. Handoff messages can be used to build patterns such as Swarm. This is useful when you want to pause the run and allow application or user to provide input when an agent hands off to them.

- SourceMatchTermination: Stops after a specific agent responds.

- ExternalTermination: Enables programmatic control of termination from outside the run. This is useful for UI integration (e.g., “Stop” buttons in chat interfaces).

- StopMessageTermination: Stops when a StopMessage is produced by an agent.

- TextMessageTermination: Stops when a TextMessage is produced by an agent.

- FunctionCallTermination: Stops when a ToolCallExecutionEvent containing a FunctionExecutionResult with a matching name is produced by an agent.

- FunctionalTermination: Stop when a function expression is evaluated to True on the last delta sequence of messages. This is useful for quickly create custom termination conditions that are not covered by the built-in ones.

## **Run Team as Streaming Mode**

In [12]:
from autogen_agentchat.base import TaskResult

task = "Write a python code that can reverse the input string."

async for message in team.run_stream(task=task):
    if isinstance(message, TaskResult):
        print("Stop Reason: ", message.stop_reason)
    else:
        print(message.source, message)

user id='35c391ca-64a8-4a7c-8839-5e8d40cc78ff' source='user' models_usage=None metadata={} created_at=datetime.datetime(2025, 7, 19, 9, 37, 23, 722724, tzinfo=datetime.timezone.utc) content='Write a python code that can reverse the input string.' type='TextMessage'
Code_Editor id='31422b0f-5a0e-40e0-a1a1-f16b1f6c87a0' source='Code_Editor' models_usage=RequestUsage(prompt_tokens=2662, completion_tokens=232) metadata={} created_at=datetime.datetime(2025, 7, 19, 9, 37, 43, 496384, tzinfo=datetime.timezone.utc) content='Here is a simple Python function to reverse an input string:\n\n```python\ndef reverse_string(s):\n    return s[::-1]\n\n# Example usage:\ninput_str = "Hello World"\nprint(reverse_string(input_str))  # Output: dlroW olleH\n```\n\nIn this code, `s[::-1]` is using Python\'s slice notation to create a reversed copy of the input string. The `::-1` part means start at the end of the string and end at position 0, move with the step -1 which means one step backwards.\n\nHowever, i

## **Agent State**

In [13]:
state = await team.save_state()
state

{'type': 'TeamState',
 'version': '1.0.0',
 'agent_states': {'Complex_DSA_solver': {'type': 'ChatAgentContainerState',
   'version': '1.0.0',
   'agent_state': {'type': 'AssistantAgentState',
    'version': '1.0.0',
    'llm_context': {'messages': [{'content': 'Write a python code like find a median two sorted array.',
       'source': 'user',
       'type': 'UserMessage'},
      {'content': 'Here is the Python code for finding the median of two sorted arrays:\n\n```python\ndef findMedian(arr1, arr2):\n    merged = sorted(arr1 + arr2)\n    n = len(merged)\n    \n    if n % 2 == 0:\n        return (merged[n // 2 - 1] + merged[n // 2]) / 2.0\n    else:\n        return merged[n // 2]\n\n# Example usage\narr1 = [1, 3]\narr2 = [2, 4]\nprint(findMedian(arr1, arr2))\n```\n\nThis code merges the two input arrays into a single sorted array and then calculates the median based on whether there is an even or odd number of elements.',
       'thought': None,
       'source': 'Complex_DSA_solver',


## **Resuming the Team**

In [14]:
from autogen_agentchat.agents import AssistantAgent

agent_1 = AssistantAgent(
    name="first_agent",
    model_client=ollama_model_clint,
    system_message="Add 1 with the number, and the first number is 0. Give the result the output."
)
agent_2 = AssistantAgent(
    name="Second_agent",
    model_client=ollama_model_clint,
    system_message="Add 1 with the number, the previous run. Give the result the output."
)
agent_3 = AssistantAgent(
    name="third_agent",
    model_client=ollama_model_clint,
    system_message="Add 1 with the number, and the previous run. Give the result the output."
)

In [15]:
team_2 = RoundRobinGroupChat(
    participants=[agent_1, agent_2, agent_3],
    max_turns=3
)

In [16]:
from autogen_agentchat.ui import Console

await Console(team_2.run_stream())

---------- TextMessage (first_agent) ----------

---------- TextMessage (Second_agent) ----------
I don't have any previous run to add 1 to. This conversation just started. What would you like to do instead?
---------- TextMessage (third_agent) ----------
How about we start fresh?

Let's play a simple game. I'll give you a number, and then you can either:

A) Add 1 to the number
B) Tell me what you'd like to do with the number (e.g., multiply it by something, divide it, etc.)

What would you like to do?


TaskResult(messages=[TextMessage(id='8c0ab86e-f768-40ef-b610-4f9af5f72b01', source='first_agent', models_usage=RequestUsage(prompt_tokens=27, completion_tokens=1), metadata={}, created_at=datetime.datetime(2025, 7, 19, 9, 52, 33, 932665, tzinfo=datetime.timezone.utc), content='', type='TextMessage'), TextMessage(id='01035638-bb11-4460-b4e4-72a85040c7cf', source='Second_agent', models_usage=RequestUsage(prompt_tokens=32, completion_tokens=27), metadata={}, created_at=datetime.datetime(2025, 7, 19, 9, 52, 36, 97598, tzinfo=datetime.timezone.utc), content="I don't have any previous run to add 1 to. This conversation just started. What would you like to do instead?", type='TextMessage'), TextMessage(id='43117f1d-f708-4014-b47d-48f9eff103e9', source='third_agent', models_usage=RequestUsage(prompt_tokens=59, completion_tokens=70), metadata={}, created_at=datetime.datetime(2025, 7, 19, 9, 52, 40, 815108, tzinfo=datetime.timezone.utc), content="How about we start fresh?\n\nLet's play a simple 

In [17]:
await Console(team_2.run_stream())

---------- TextMessage (first_agent) ----------
Let's start fresh.

In that case, I'll choose option B). What number will you give me, and what operation would you like me to perform on it?

(Also, feel free to change your mind about the "add 1" option if you'd rather stick with it)
---------- TextMessage (Second_agent) ----------
Let's play a game with numbers.

Here's a random number: 42

What I'd like you to do is... multiply it by 2!

Your turn!
---------- TextMessage (third_agent) ----------
I'll make sure to keep things flexible from now on.

Okay, multiplying 42 by 2 gives us:

84

Now it's my turn!

I'd like you to perform a simple operation on the result. Let's say... divide 84 by 4.

Your turn again!


TaskResult(messages=[TextMessage(id='dc1ef9f9-25cd-4e17-9ad4-65e55df7e32f', source='first_agent', models_usage=RequestUsage(prompt_tokens=136, completion_tokens=60), metadata={}, created_at=datetime.datetime(2025, 7, 19, 9, 53, 46, 263160, tzinfo=datetime.timezone.utc), content='Let\'s start fresh.\n\nIn that case, I\'ll choose option B). What number will you give me, and what operation would you like me to perform on it?\n\n(Also, feel free to change your mind about the "add 1" option if you\'d rather stick with it)', type='TextMessage'), TextMessage(id='8ff0d1ef-fa13-4e01-8b03-e83e930eab13', source='Second_agent', models_usage=RequestUsage(prompt_tokens=196, completion_tokens=36), metadata={}, created_at=datetime.datetime(2025, 7, 19, 9, 53, 48, 981712, tzinfo=datetime.timezone.utc), content="Let's play a game with numbers.\n\nHere's a random number: 42\n\nWhat I'd like you to do is... multiply it by 2!\n\nYour turn!", type='TextMessage'), TextMessage(id='935fe11d-ac7a-4ea7-9176-27f7

## **State Reset and re-run again**

In [18]:
await team_2.reset()
await Console(team_2.run_stream())

---------- TextMessage (first_agent) ----------

---------- TextMessage (Second_agent) ----------
I don't have any prior runs or numbers to work with. This conversation just started. If you'd like to provide a number and I'll add 1 to it, I can do that!
---------- TextMessage (third_agent) ----------
Let's start fresh.

What number would you like me to add 1 to? You can type any number you'd like, and I'll give you the result!


TaskResult(messages=[TextMessage(id='08ffdd89-f828-455a-95a1-c6e66c9bb773', source='first_agent', models_usage=RequestUsage(prompt_tokens=27, completion_tokens=1), metadata={}, created_at=datetime.datetime(2025, 7, 19, 9, 54, 46, 538999, tzinfo=datetime.timezone.utc), content='', type='TextMessage'), TextMessage(id='e35e3939-3f58-477c-a512-63f6498561b8', source='Second_agent', models_usage=RequestUsage(prompt_tokens=32, completion_tokens=41), metadata={}, created_at=datetime.datetime(2025, 7, 19, 9, 54, 49, 625181, tzinfo=datetime.timezone.utc), content="I don't have any prior runs or numbers to work with. This conversation just started. If you'd like to provide a number and I'll add 1 to it, I can do that!", type='TextMessage'), TextMessage(id='de8510fc-5408-4594-af07-97fd05d6b313', source='third_agent', models_usage=RequestUsage(prompt_tokens=73, completion_tokens=35), metadata={}, created_at=datetime.datetime(2025, 7, 19, 9, 54, 52, 111814, tzinfo=datetime.timezone.utc), content="Le